In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%%RecordEvent
# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
#from tpch import load_lineitem, load_orders, load_supplier, load_nation, q21
import pandas as pd
DATA_ROOT = "/home/jupyter/tpch-dbgen/data"
STORAGE_OPTS = {}



In [ ]:

### cell 0 ###

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)


In [ ]:

### cell 1 ###

def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)


In [ ]:

### cell 2 ###

def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)


In [ ]:

### cell 3 ###

def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [ ]:

# %%time
# ### cell 4 ###

# # Filter lineitems where receipt date > commit date
# mask_date = lineitem.L_RECEIPTDATE > lineitem.L_COMMITDATE
# fi = lineitem[mask_date]

# # Compute unique supplier counts per order
# orig_count = (
#     lineitem
#     .groupby('L_ORDERKEY', sort=False)
#     .agg({'L_SUPPKEY': 'nunique'})
#     .reset_index()
#     .rename(columns={'L_SUPPKEY': 'orig_count'})
# )
# after_count = (
#     fi
#     .groupby('L_ORDERKEY', sort=False)
#     .agg({'L_SUPPKEY': 'nunique'})
#     .reset_index()
#     .rename(columns={'L_SUPPKEY': 'after_count'})
# )

# # Identify valid orders (orig_count >1 and after_count ==1)
# valid_orders = orig_count.merge(after_count, on='L_ORDERKEY', how='inner')
# valid_orders = valid_orders[(valid_orders.orig_count > 1) & (valid_orders.after_count == 1)]['L_ORDERKEY']

# # Restrict to orders with status 'F'
# f_orders = orders[orders.O_ORDERSTATUS == 'F']['O_ORDERKEY']
# valid_orders = valid_orders[valid_orders.isin(f_orders)]

# # Pre-filter Saudi Arabian suppliers
# supplier_sa = supplier.merge(
#     nation[nation.N_NAME == 'SAUDI ARABIA'][['N_NATIONKEY']],
#     left_on='S_NATIONKEY',
#     right_on='N_NATIONKEY',
#     how='inner'
# )[['S_SUPPKEY', 'S_NAME']]

# # Final aggregation: count waiting lineitems per Saudi supplier
# total = (
#     lineitem[mask_date & lineitem.L_ORDERKEY.isin(valid_orders)]
#     .merge(supplier_sa, left_on='L_SUPPKEY', right_on='S_SUPPKEY', how='inner')
#     .groupby('S_NAME', as_index=False, sort=False)
#     .size()
#     .rename(columns={'size': 'NUMWAIT'})
#     .sort_values(['NUMWAIT', 'S_NAME'], ascending=[False, True])
# )

In [ ]:
%%time
#original
lineitem_filtered = lineitem.loc[
    :, ["L_ORDERKEY", "L_SUPPKEY", "L_RECEIPTDATE", "L_COMMITDATE"]
]

# Keep all rows that have another row in linetiem with the same orderkey and different suppkey
lineitem_orderkeys = (
    lineitem_filtered.loc[:, ["L_ORDERKEY", "L_SUPPKEY"]]
    .groupby("L_ORDERKEY", as_index=False, sort=False)["L_SUPPKEY"]
    .nunique()
)
lineitem_orderkeys.columns = ["L_ORDERKEY", "nunique_col"]
lineitem_orderkeys = lineitem_orderkeys[lineitem_orderkeys["nunique_col"] > 1]
lineitem_orderkeys = lineitem_orderkeys.loc[:, ["L_ORDERKEY"]]

# Keep all rows that have l_receiptdate > l_commitdate
lineitem_filtered = lineitem_filtered[
    lineitem_filtered["L_RECEIPTDATE"] > lineitem_filtered["L_COMMITDATE"]
]
lineitem_filtered = lineitem_filtered.loc[:, ["L_ORDERKEY", "L_SUPPKEY"]]

# Merge Filter + Exists
lineitem_filtered = lineitem_filtered.merge(
    lineitem_orderkeys, on="L_ORDERKEY", how="inner"
)

# Not Exists: Check the exists condition isn't still satisfied on the output.
lineitem_orderkeys = lineitem_filtered.groupby(
    "L_ORDERKEY", as_index=False, sort=False
)["L_SUPPKEY"].nunique()
lineitem_orderkeys.columns = ["L_ORDERKEY", "nunique_col"]
lineitem_orderkeys = lineitem_orderkeys[lineitem_orderkeys["nunique_col"] == 1]
lineitem_orderkeys = lineitem_orderkeys.loc[:, ["L_ORDERKEY"]]

# Merge Filter + Not Exists
lineitem_filtered = lineitem_filtered.merge(
    lineitem_orderkeys, on="L_ORDERKEY", how="inner"
)

orders_filtered = orders.loc[:, ["O_ORDERSTATUS", "O_ORDERKEY"]]
orders_filtered = orders_filtered[orders_filtered["O_ORDERSTATUS"] == "F"]
orders_filtered = orders_filtered.loc[:, ["O_ORDERKEY"]]
total = lineitem_filtered.merge(
    orders_filtered, left_on="L_ORDERKEY", right_on="O_ORDERKEY", how="inner"
)
total = total.loc[:, ["L_SUPPKEY"]]

supplier_filtered = supplier.loc[:, ["S_SUPPKEY", "S_NATIONKEY", "S_NAME"]]
total = total.merge(
    supplier_filtered, left_on="L_SUPPKEY", right_on="S_SUPPKEY", how="inner"
)
total = total.loc[:, ["S_NATIONKEY", "S_NAME"]]
nation_filtered = nation.loc[:, ["N_NAME", "N_NATIONKEY"]]
nation_filtered = nation_filtered[nation_filtered["N_NAME"] == "SAUDI ARABIA"]
total = total.merge(
    nation_filtered, left_on="S_NATIONKEY", right_on="N_NATIONKEY", how="inner"
)
total = total.loc[:, ["S_NAME"]]
total = total.groupby("S_NAME", as_index=False, sort=False).size()
total.columns = ["S_NAME", "NUMWAIT"]
total = total.sort_values(by=["NUMWAIT", "S_NAME"], ascending=[False, True])


In [10]:
total

,S_NAME,NUMWAIT
946,Supplier#000062538,24
100,Supplier#000032858,22
2627,Supplier#000063723,21
2354,Supplier#000089484,21
972,Supplier#000007061,20
...,...,...
1836,Supplier#000006598,2
3888,Supplier#000016485,2
3582,Supplier#000031729,2
3696,Supplier#000098641,2


In [9]:
total

,S_NAME,NUMWAIT
2507,Supplier#000062538,24
1350,Supplier#000032858,22
2555,Supplier#000063723,21
3583,Supplier#000089484,21
293,Supplier#000007061,20
...,...,...
265,Supplier#000006598,2
675,Supplier#000016485,2
1309,Supplier#000031729,2
3940,Supplier#000098641,2


In [10]:
### cell 5 ###

total.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 4009 entries, 2507 to 3448
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   S_NAME   4009 non-null   object
 1   NUMWAIT  4009 non-null   int64
dtypes: int64(1), object(1)
memory usage: 148.8+ KB
